In [2]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
from ast import literal_eval
from collections import Counter
import math
import numpy as np
import seaborn as sns

In [3]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [4]:
benchmarks = [
    # High-sharing Apps
    'bfs',
    'gemm',
    'lu',
    'spmv',
    'mm3',
    'mm2',
    'matrixmultiplication',
    'syr2k',
    'gups',    
    'mis',
    'matrixtranspose',
    'stencil2d',
    'gaussian',
    'im2col',
    'vgg16',

    # Low-sharing Apps
    'fdtd2d',
    'fastwalshtransform',
    'jacobi1d',
    'relu',
    'color',
    'convolution2d',
    'sssp',
    'fir',
    'fft',
    'bitonicsort',
    'pagerank',
]

In [5]:
num_chiplets = 36
ns = 0.000000001
chip_cols = [str(i) for i in range(36)]

In [6]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = "20260221-223953"
expanded_dir = f"{base_dir}/{date}/expanded"
trace_dir = f"{base_dir}/{date}"
print(f"{base_dir}/{date}")

/home/ehdgus7279/mgvm/results/memory-traces/20260221-223953


In [6]:
# benchmark = 'syr2k'
benchmark = 'bfs'

df = pd.read_csv(f"../pc-rdma-metrics/{benchmark}_pc_rdma_metrics.csv", 
                skipinitialspace=True)

df['shared_access_count_ratio'] = df['shared_access_count'] / df['interval_total_access']

df

,interval,pc,shared_cacheline_count,shared_access_count,shared_access_mean,shared_access_std,shared_shared_degree_mean,shared_shared_degree_std,shared_cacheline_ratio,shared_access_ratio,...,private_access_std,private_shared_degree_mean,private_shared_degree_std,private_cacheline_ratio,private_access_ratio,pc_total_cachelines,pc_total_access,interval_total_cachelines,interval_total_access,shared_access_count_ratio
0,0.0,1061167752,13386.0,13386.0,1.0000,0.000000,1.0,0.0,0.287883,0.284107,...,0.0,0.0,0.0,0.711644,0.702309,46476,46476,46498,47116,0.284107
1,0.0,1061167360,1.0,288.0,288.0000,0.000000,35.0,0.0,0.000022,0.006113,...,0.0,0.0,0.0,0.000000,0.000000,1,288,46498,47116,0.006113
2,0.0,1061167396,1.0,288.0,288.0000,0.000000,35.0,0.0,0.000022,0.006113,...,0.0,0.0,0.0,0.000000,0.000000,1,288,46498,47116,0.006113
3,0.0,1061167796,0.0,0.0,0.0000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.000022,0.000021,1,1,46498,47116,0.000000
4,0.0,1061167868,0.0,0.0,0.0000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.000344,0.000340,16,16,46498,47116,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,760000.0,1061192508,16.0,19.0,1.1875,0.403113,1.0,0.0,0.695652,0.441860,...,0.0,0.0,0.0,0.000000,0.000000,16,19,23,43,0.441860
356,760000.0,1061192476,12.0,12.0,1.0000,0.000000,1.0,0.0,0.521739,0.279070,...,0.0,0.0,0.0,0.000000,0.000000,12,12,23,43,0.279070
357,760000.0,1061192512,2.0,2.0,1.0000,0.000000,1.0,0.0,0.086957,0.046512,...,0.0,0.0,0.0,0.000000,0.000000,2,2,23,43,0.046512
358,760000.0,1061192444,0.0,0.0,0.0000,0.000000,0.0,0.0,0.000000,0.000000,...,0.0,0.0,0.0,0.217391,0.116279,5,5,23,43,0.000000


In [7]:
trace_df = pd.read_csv(f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv")
trace_df

,time,pc,addr,owner,rw,0,1,2,3,4,...,28,29,30,31,32,33,34,35,access_count,shared_chiplets
0,0.043098,1061167360,3211542528,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
1,0.043098,1061167360,3211542528,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
2,0.043098,1061167360,3211542528,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
3,0.043098,1061167360,3211542528,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,2,1
4,0.043098,1061167360,3211542528,0,0,0,2,0,0,0,...,0,0,0,0,0,0,0,0,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
552783,0.043860,1061192528,3211534336,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
552784,0.043861,1061192508,2149815936,35,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
552785,0.043861,1061192508,2149816000,35,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
552786,0.043861,1061192508,2149816064,35,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1


In [8]:
clusters_3x3 = {
    'cluster_0_3x3': [0, 1, 2, 6, 7, 8, 12, 13, 14],
    'cluster_1_3x3': [3, 4, 5, 9, 10, 11, 15, 16, 17],
    'cluster_2_3x3': [18, 19, 20, 24, 25, 26, 30, 31, 32],
    'cluster_3_3x3': [21, 22, 23, 27, 28, 29, 33, 34, 35],
}

for cluster_name, chiplet_ids in clusters_3x3.items():
    str_cols = [str(x) for x in chiplet_ids]
    
    trace_df[cluster_name] = trace_df[str_cols].sum(axis=1)

In [9]:
clusters_2x2 = {
    'cluster_0_2x2': [0, 1, 6, 7],       # 좌측 상단
    'cluster_1_2x2': [2, 3, 8, 9],       # 상단 중앙
    'cluster_2_2x2': [4, 5, 10, 11],     # 우측 상단
    'cluster_3_2x2': [12, 13, 18, 19],   # 좌측 중앙
    'cluster_4_2x2': [14, 15, 20, 21],   # 정중앙
    'cluster_5_2x2': [16, 17, 22, 23],   # 우측 중앙
    'cluster_6_2x2': [24, 25, 30, 31],   # 좌측 하단
    'cluster_7_2x2': [26, 27, 32, 33],   # 하단 중앙
    'cluster_8_2x2': [28, 29, 34, 35]    # 우측 하단
}

for cluster_name, chiplet_ids in clusters_2x2.items():
    str_cols = [str(x) for x in chiplet_ids]
    
    trace_df[cluster_name] = trace_df[str_cols].sum(axis=1)

In [10]:
clusters_3x3_cols = [k for k in clusters_3x3.keys()]
clusters_2x2_cols = [k for k in clusters_2x2.keys()]

chiplet_to_cluster_2x2 = {}
for cluster_name, chiplet_ids in clusters_2x2.items():
    for chip_id in chiplet_ids:
        chiplet_to_cluster_2x2[chip_id] = cluster_name

chiplet_to_cluster_3x3 = {}
for cluster_name, chiplet_ids in clusters_3x3.items():
    for chip_id in chiplet_ids:
        chiplet_to_cluster_3x3[chip_id] = cluster_name

In [11]:
t_min = trace_df['time'].min()
trace_df['norm_time'] = (trace_df['time'] - t_min) / ns
trace_df['interval'] = (trace_df['norm_time'] // 10000) * 10000

In [12]:
agg_dict = {col: 'sum' for col in clusters_3x3}
agg_dict.update({col: 'sum' for col in clusters_2x2})
agg_dict['rw'] = 'max'     # 1이 하나라도 있으면 RW 캐시라인
agg_dict['owner'] = 'first' # owner는 동일하므로 하나만 취함

interval_addr_df = trace_df.groupby(['interval', 'addr'], as_index=False).agg(agg_dict)

In [13]:
interval_addr_df['owner_cluster_2x2'] = interval_addr_df['owner'].map(chiplet_to_cluster_2x2)
interval_addr_df['owner_cluster_3x3'] = interval_addr_df['owner'].map(chiplet_to_cluster_3x3)

In [14]:
owner_vals_2x2 = np.array(interval_addr_df['owner_cluster_2x2'], dtype=str)[:, None]
col_vals_2x2 = np.array(clusters_2x2_cols, dtype=str)
is_owner_2x2 = owner_vals_2x2 == col_vals_2x2
interval_addr_df[clusters_2x2_cols] = interval_addr_df[clusters_2x2_cols].mask(is_owner_2x2, 0)

owner_vals_3x3 = np.array(interval_addr_df['owner_cluster_3x3'], dtype=str)[:, None]
col_vals_3x3 = np.array(clusters_3x3_cols, dtype=str)
is_owner_3x3 = owner_vals_3x3 == col_vals_3x3
interval_addr_df[clusters_3x3_cols] = interval_addr_df[clusters_3x3_cols].mask(is_owner_3x3, 0)

In [18]:
interval_addr_df['is_shared_2x2'] = (interval_addr_df[clusters_2x2_cols] != 0).any(axis=1)
interval_addr_df['shared_degree_2x2'] = (interval_addr_df[clusters_2x2_cols] != 0).sum(axis=1)

interval_addr_df['is_shared_3x3'] = (interval_addr_df[clusters_3x3_cols] != 0).any(axis=1)
interval_addr_df['shared_degree_3x3'] = (interval_addr_df[clusters_3x3_cols] != 0).sum(axis=1)

In [19]:
interval_addr_df.columns

Index(['interval', 'addr', 'cluster_0_3x3', 'cluster_1_3x3', 'cluster_2_3x3',
       'cluster_3_3x3', 'cluster_0_2x2', 'cluster_1_2x2', 'cluster_2_2x2',
       'cluster_3_2x2', 'cluster_4_2x2', 'cluster_5_2x2', 'cluster_6_2x2',
       'cluster_7_2x2', 'cluster_8_2x2', 'rw', 'owner', 'owner_cluster_2x2',
       'owner_cluster_3x3', 'is_shared_2x2', 'sharing_degree_2x2',
       'is_shared_3x3', 'sharing_degree_3x3', 'shared_degree_2x2',
       'shared_degree_3x3'],
      dtype='str')

In [20]:
interval_addr_df

,interval,addr,cluster_0_3x3,cluster_1_3x3,cluster_2_3x3,cluster_3_3x3,cluster_0_2x2,cluster_1_2x2,cluster_2_2x2,cluster_3_2x2,...,rw,owner,owner_cluster_2x2,owner_cluster_3x3,is_shared_2x2,sharing_degree_2x2,is_shared_3x3,sharing_degree_3x3,shared_degree_2x2,shared_degree_3x3
0,0.0,2147487744,0,0,0,0,0,0,0,0,...,1,0,cluster_0_2x2,cluster_0_3x3,False,0,False,0,0,0
1,0.0,2147487808,0,0,0,0,0,0,0,0,...,1,0,cluster_0_2x2,cluster_0_3x3,False,0,False,0,0,0
2,0.0,2147487872,0,0,0,0,0,0,0,0,...,1,0,cluster_0_2x2,cluster_0_3x3,False,0,False,0,0,0
3,0.0,2147487936,0,0,0,0,0,0,0,0,...,1,0,cluster_0_2x2,cluster_0_3x3,False,0,False,0,0,0
4,0.0,2147488000,0,0,0,0,0,0,0,0,...,1,0,cluster_0_2x2,cluster_0_3x3,False,0,False,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347744,760000.0,2234782656,0,0,0,0,0,0,0,0,...,0,0,cluster_0_2x2,cluster_0_3x3,False,0,False,0,0,0
347745,760000.0,2305611328,0,0,0,0,0,0,0,0,...,0,0,cluster_0_2x2,cluster_0_3x3,False,0,False,0,0,0
347746,760000.0,2305611392,0,0,0,0,0,0,0,0,...,0,0,cluster_0_2x2,cluster_0_3x3,False,0,False,0,0,0
347747,760000.0,2305611456,0,0,0,0,0,0,0,0,...,0,0,cluster_0_2x2,cluster_0_3x3,False,0,False,0,0,0
